#IMDB reviews - Sentiment Analysis
using LSTM(Long Short-Term Memory layer)

install kaggle & Get the dataSet : https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [ ]:
!pip install kaggle

**Importing the Dependencies**

In [ ]:
import os
import json

import pandas as pd
from zipfile import ZipFile
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

* Embedding: Converts words/tokens (integers) into dense vector representations.
* LSTM: Long Short-Term Memory layer — captures patterns and dependencies in sequences (like sentences).
* Dense: Fully connected layer used at the output (e.g., for binary classification).
* pad_sequences : Ensures that all sequences are the same length by padding shorter ones (or truncating longer ones).
* Tokenizer: Converts raw text into sequences of integers (token IDs).

##Data Collection- Kaggle API

In [ ]:
kaggle_dict = json.load(open("kaggle.json"))

In [ ]:
kaggle_dict.keys()

dict_keys(['username', 'key'])

In [ ]:
#Settingup kaggle user credential inside environment Variable
os.environ["KAGGLE_USERNAME"] = kaggle_dict["username"]
os.environ["KAGGLE_KEY"] = kaggle_dict["key"]

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other


In [ ]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


In [ ]:
# unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
  zip_ref.extractall()

In [ ]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


##Load the Dataset

In [ ]:
#load the dataset in DataFrame
data = pd.read_csv("/content/IMDB Dataset.csv")

In [ ]:
data.shape

(50000, 2)

In [ ]:
data.sample(5)

,review,sentiment
9920,I watched pp the other night. I have to say I ...,positive
27590,Recently had the pleasure of seeing this emoti...,positive
36486,I first saw this film when it was transmitted ...,positive
7845,Once when I was in college and we had an inter...,positive
8771,"I think this is a great, classic monster film ...",positive


In [ ]:
#check the values in the dataset
data['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


Observation :  This is the balanced dataset of IMDB reviews

In [ ]:
#Converted sentiment labels from text to binary
data.replace({"sentiment" : {"positive" : 1 ,"negative" : 0}}, inplace=True)
data.sample(5)

,review,sentiment
40843,I just finished this movie and my only comment...,1
34057,This movie is an almost forgotten gem from 197...,1
15206,"Really, really bad slasher movie. A psychotic ...",0
23572,Its perhaps unfair of me to comment on this fi...,0
33091,Fun With Dick and Jane failed to entertain on ...,0


In [ ]:
data["sentiment"].value_counts()

,count
sentiment,
1,25000
0,25000


##Train Test Split Data

In [ ]:
# split data into training data and test data
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


###Data Preprocessing

In [ ]:
# Tokenize text data
tokenizer = Tokenizer(num_words=5000)

#Fits the tokenizer on the training data only to avoid data leakage.
tokenizer.fit_on_texts(train_data['review'])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]),maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]),maxlen=200)

In [ ]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [ ]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [ ]:
y_train = train_data["sentiment"]
y_test = test_data["sentiment"]

In [ ]:
print(y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


In [ ]:
print(y_test)

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


##LSTM - Long Short-Term Memory

In [ ]:
# build the model

model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

* Embedding Layer: Converts word indices to dense 128-dimensional vectors.
* LSTM Layer: Captures long-term dependencies in the text; 128 memory units.
* Dropout: Regularizes both input and recurrent connections (helps prevent overfitting).
* Dense Output Layer: sigmoid activation outputs probability (0 to 1) for binary classification.

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

* Adam: Adaptive learning rate optimizer — very effective for NLP tasks.
* Binary Crossentropy: The correct loss function for binary classification.
* Accuracy: Used as the evaluation metric during training and testing.

###Training the Model

In [ ]:
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 193s 371ms/step - accuracy: 0.7126 - loss: 0.5392 - val_accuracy: 0.8414 - val_loss: 0.3554
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 191s 382ms/step - accuracy: 0.8527 - loss: 0.3537 - val_accuracy: 0.8499 - val_loss: 0.3505
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 201s 380ms/step - accuracy: 0.8835 - loss: 0.2927 - val_accuracy: 0.8621 - val_loss: 0.3279
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 381ms/step - accuracy: 0.8992 - loss: 0.2549 - val_accuracy: 0.8740 - val_loss: 0.3229
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 203s 383ms/step - accuracy: 0.9124 - loss: 0.2208 - val_accuracy: 0.8741 - val_loss: 0.3225


###Model Evaluation

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 111ms/step - accuracy: 0.8742 - loss: 0.3130
Test Loss: 0.30611056089401245
Test Accuracy: 0.8812000155448914


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, 200, 128)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (64, 128)              │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,315,141 (8.83 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,543,428 (5.89 MB)

##Building a Predictive System

In [ ]:
def predict_sentiment(review):
  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [ ]:
# example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step
The sentiment of the review is: positive


In [ ]:
# example usage
new_review = "This movie was not that good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
The sentiment of the review is: negative


In [ ]:
# example usage
new_review = "This movie was ok but not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
The sentiment of the review is: negative


In [ ]:
# example usage
new_review = "This movie was very very very good that i wont even recommend to watch this waste of money "
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
The sentiment of the review is: negative
